# ISA95 Quality Check (Fabric Notebook)
Run ISA95 CSV validation inside Microsoft Fabric using the reusable validator script.

In [ ]:
LAKEHOUSE_ROOT = globals().get("LAKEHOUSE_ROOT", "/lakehouse/default")
ZIP_PATH = f"{LAKEHOUSE_ROOT}/Files/isa95_data_zip/migration_2026-04-17T07-57-10.065Z.zip"
EXTRACT_TO = f"{LAKEHOUSE_ROOT}/Files/isa95/isa95_data"

if Path(ZIP_PATH).exists():
    extracted_files = unzip_file(ZIP_PATH, EXTRACT_TO, overwrite=False)
    extracted_files[:10]
else:
    print(f"Update ZIP_PATH first. File not found: {ZIP_PATH}")

In [ ]:
from pathlib import Path
import zipfile


def unzip_file(zip_path: str, extract_to: str, overwrite: bool = False) -> list[str]:
    """Unzip a .zip file in Fabric/Lakehouse and return extracted file paths."""
    zip_file = Path(zip_path)
    target_dir = Path(extract_to)

    if not zip_file.exists():
        raise FileNotFoundError(f"ZIP file not found: {zip_file}")

    if zip_file.suffix.lower() != ".zip":
        raise ValueError(f"Expected a .zip file, got: {zip_file}")

    target_dir.mkdir(parents=True, exist_ok=True)

    extracted: list[str] = []
    with zipfile.ZipFile(zip_file, "r") as zf:
        for member in zf.infolist():
            output_path = target_dir / member.filename

            if member.is_dir():
                output_path.mkdir(parents=True, exist_ok=True)
                continue

            output_path.parent.mkdir(parents=True, exist_ok=True)

            # Keep existing files unless overwrite=True.
            if output_path.exists() and not overwrite:
                extracted.append(str(output_path))
                continue

            with zf.open(member, "r") as source, open(output_path, "wb") as dest:
                dest.write(source.read())

            extracted.append(str(output_path))

    print(f"Extracted {len(extracted)} files to: {target_dir}")
    return extracted

## 1) Configure Paths
Update these paths for your Lakehouse workspace.

In [ ]:
LAKEHOUSE_ROOT = "/lakehouse/default"

# Pfade anpassen:
DATA_DIR = f"{LAKEHOUSE_ROOT}/Files/isa95_data"                     # Pfad zu den migrierten CSV-Dateien
SCRIPTS_DIR = f"{LAKEHOUSE_ROOT}/Files/isa95/isa95_script"          # Ordner mit Validator-Script
VALIDATOR_SCRIPT = f"{SCRIPTS_DIR}/validate_isa95_entities_and_mappings.py"
METADATA_CACHE = f"{LAKEHOUSE_ROOT}/Files/isa95/config/dtdl_metadata_cache.json"
VALUE_RULES_CONFIG = f"{LAKEHOUSE_ROOT}/Files/isa95/config/quality-value-rules.json"
OUTPUT_JSON = f"{LAKEHOUSE_ROOT}/Files/isa95_quality_output/quality-check-report.json"

DATA_DIRS = [DATA_DIR]

# Nur nötig wenn refresh_metadata=True (DTDL-Dateien im Lakehouse vorhanden)
DTDL_DIR = f"{LAKEHOUSE_ROOT}/Files/InbuiltEntitiesDTDL"

# Referential Integrity Mode:
#   "python" → in-memory Python check (default, no Warehouse needed)
#   "sql"    → Fabric SQL Warehouse anti-join (not yet implemented, reserved for future)
RI_MODE = "python"

# Max rows per file for mapping + non-referenced entity CSVs (Python mode).
# Referenced entity tables always load all rows (PK+timestamp columns only).
# Set to None to load all rows (may cause OOM on large datasets).
MAX_ROWS_PER_FILE = 50_000

print(f"RI_MODE: {RI_MODE}")
print(f"MAX_ROWS_PER_FILE: {MAX_ROWS_PER_FILE}")
print(f"OUTPUT_JSON: {OUTPUT_JSON}")

## 2) Import Validator

In [ ]:
import sys
import importlib.util
from pathlib import Path

if SCRIPTS_DIR not in sys.path:
    sys.path.append(SCRIPTS_DIR)

validator_path = Path(VALIDATOR_SCRIPT)
if not validator_path.exists():
    raise FileNotFoundError(f"Validator script not found: {validator_path}")

spec = importlib.util.spec_from_file_location("isa95_validator", str(validator_path))
if spec is None or spec.loader is None:
    raise ImportError(f"Cannot load validator module from: {validator_path}")

validator_module = importlib.util.module_from_spec(spec)
# Register module before exec so dataclass/type introspection can resolve __module__ correctly.
sys.modules[spec.name] = validator_module
spec.loader.exec_module(validator_module)
run_validation_for_notebook = validator_module.run_validation_for_notebook
print_report_summary = validator_module.print_report_summary

print(f"Python path contains scripts dir: {SCRIPTS_DIR in sys.path}")
print(f"Scripts dir exists: {Path(SCRIPTS_DIR).exists()}")
print(f"Validator script exists: {validator_path.exists()} -> {validator_path}")

## 3) Run Validation
Set `refresh_metadata=True` on first run or after DTDL changes.

In [ ]:
import json as _json
from pathlib import Path
from datetime import datetime, timezone

cache_exists = Path(METADATA_CACHE).exists()
dtdl_exists  = Path(DTDL_DIR).exists()
refresh_metadata = False

print(f"DATA_DIR exists:      {Path(DATA_DIR).exists()} -> {DATA_DIR}")
print(f"SCRIPTS_DIR exists:   {Path(SCRIPTS_DIR).exists()} -> {SCRIPTS_DIR}")
print(f"VALIDATOR_SCRIPT exists:{Path(VALIDATOR_SCRIPT).exists()} -> {VALIDATOR_SCRIPT}")
print(f"METADATA_CACHE exists:{cache_exists} -> {METADATA_CACHE}")
print(f"DTDL_DIR exists:      {dtdl_exists} -> {DTDL_DIR}")
print(f"RI_MODE:              {RI_MODE}")

value_rules_path = VALUE_RULES_CONFIG if Path(VALUE_RULES_CONFIG).exists() else ""
if value_rules_path:
    print(f"VALUE_RULES_CONFIG exists: {value_rules_path}")
else:
    print(f"VALUE_RULES_CONFIG not found, defaults only: {VALUE_RULES_CONFIG}")

if not cache_exists and not dtdl_exists:
    raise FileNotFoundError(
        "Neither metadata cache nor DTDL directory exists. "
        "Upload dtdl_metadata_cache.json to the configured METADATA_CACHE path "
        "or upload DTDL JSON files to Files/InbuiltEntitiesDTDL."
    )
if not cache_exists and dtdl_exists:
    print("Metadata cache not found — building from DTDL directory...")
    refresh_metadata = True

# ── Python in-memory mode ─────────────────────────────────────────────────
if RI_MODE == "python":
    print(f"Max rows/file (mapping + non-referenced): {MAX_ROWS_PER_FILE}")
    report = run_validation_for_notebook(
        data_dirs=DATA_DIRS,
        dtdl_dir=DTDL_DIR if refresh_metadata else None,
        metadata_cache=METADATA_CACHE,
        refresh_metadata=refresh_metadata,
        output_json=OUTPUT_JSON,
        include_master_process=False,
        max_rows_per_file=MAX_ROWS_PER_FILE,
        value_rules_config=value_rules_path,
    )

# ── Spark SQL mode ────────────────────────────────────────────────────────
elif RI_MODE == "sql":
    import glob, re, csv as _csv, io

    STAGING_DB = "ri_staging"    # temporary Spark database (dropped & recreated each run)

    # 1) Drop and recreate staging database (clears all previous staging tables)
    print(f"Dropping staging database '{STAGING_DB}' ...")
    spark.sql(f"DROP DATABASE IF EXISTS {STAGING_DB} CASCADE")
    spark.sql(f"CREATE DATABASE {STAGING_DB}")
    print(f"Staging database '{STAGING_DB}' recreated.")

    def _normalize(name: str) -> str:
        return re.sub(r"[^a-z0-9]", "", name.lower())

    def _csv_to_spark_table(csv_path: str, table_name: str, keep_cols=None) -> int:
        """Load a CSV into a Spark staging table; return row count."""
        df = spark.read.option("header", "true").option("encoding", "UTF-8").csv(csv_path)
        if keep_cols:
            existing = [c for c in keep_cols if c in df.columns]
            if existing:
                df = df.select(*existing)
        df.write.mode("overwrite").saveAsTable(f"{STAGING_DB}.{table_name}")
        return df.count()

    # 2) Discover and load mapping files (limited to MAX_ROWS_PER_FILE via sample)
    mapping_cols_required = {"sourceType", "sourcePrimaryKey", "targetType", "targetPrimaryKey"}
    csv_files = glob.glob(f"{DATA_DIR}/**/*.csv", recursive=True)

    mapping_files = []
    entity_files  = []
    for f in csv_files:
        try:
            with open(f, "r", encoding="utf-8-sig") as fh:
                header_line = fh.readline()
            headers = {h.strip() for h in header_line.split(",")}
            if mapping_cols_required.issubset(headers):
                mapping_files.append(f)
            else:
                entity_files.append(f)
        except Exception:
            pass

    print(f"Found {len(mapping_files)} mapping file(s), {len(entity_files)} entity file(s).")

    # 3) Load mapping files; collect referenced entity types
    referenced_types = set()
    mapping_tables_loaded = []

    for mf in mapping_files:
        tbl = "mapping_" + _normalize(Path(mf).stem)[:60]
        count = _csv_to_spark_table(mf, tbl)
        mapping_tables_loaded.append((mf, tbl))
        # Collect all sourceType / targetType values
        for row in spark.sql(f"SELECT DISTINCT sourceType, targetType FROM {STAGING_DB}.{tbl}").collect():
            if row["sourceType"]: referenced_types.add(_normalize(row["sourceType"]))
            if row["targetType"]: referenced_types.add(_normalize(row["targetType"]))
        print(f"  Loaded mapping: {Path(mf).name} -> {tbl} ({count} rows)")

    print(f"Referenced entity types: {sorted(referenced_types)}")

    # 4) Load only referenced entity files (PrimaryKey + sourceTimestamp only, ALL rows)
    entity_pk_tables = {}   # normalized_type -> spark table name

    for ef in entity_files:
        stem_norm = _normalize(Path(ef).stem.rstrip("s"))   # simple singularize
        if stem_norm not in referenced_types and _normalize(Path(ef).stem) not in referenced_types:
            continue
        tbl = "entity_" + _normalize(Path(ef).stem)[:60]
        # Read only RI-relevant columns
        try:
            with open(ef, "r", encoding="utf-8-sig") as fh:
                headers_raw = next(_csv.DictReader(fh)).keys()
            keep = [h for h in headers_raw if _normalize(h) in {"sourcetimestamp", "primarykey"}]
            if not keep:
                keep = None  # keep all if none detected
        except Exception:
            keep = None
        count = _csv_to_spark_table(ef, tbl, keep_cols=keep)
        for t in [_normalize(Path(ef).stem), _normalize(Path(ef).stem.rstrip("s"))]:
            entity_pk_tables[t] = tbl
        print(f"  Loaded entity:  {Path(ef).name} -> {tbl} ({count} rows, pk_only)")

    # 5) Run anti-join RI checks per mapping table
    ri_issues = []

    for mf, mtbl in mapping_tables_loaded:
        for side, type_col, pk_col in [
            ("source", "sourceType", "sourcePrimaryKey"),
            ("target", "targetType", "targetPrimaryKey"),
        ]:
            # Get distinct types in this mapping file for this side
            types_in_mapping = spark.sql(
                f"SELECT DISTINCT {type_col} FROM {STAGING_DB}.{mtbl} WHERE {type_col} IS NOT NULL"
            ).collect()

            for row in types_in_mapping:
                etype = row[type_col]
                enorm = _normalize(etype)
                etbl  = entity_pk_tables.get(enorm) or entity_pk_tables.get(enorm + "")

                if not etbl:
                    ri_issues.append({
                        "severity": "warning",
                        "rule": f"mapping_{side}_entity_unresolved",
                        "file": mf,
                        "message": f"Could not resolve {side} entity type '{etype}' to a loaded CSV",
                        "details": {f"{side}Type": etype},
                    })
                    continue

                # Anti-join: mapping rows whose PK is not in the entity table PrimaryKey
                entity_columns = spark.table(f"{STAGING_DB}.{etbl}").columns
                entity_pk_col = next((c for c in entity_columns if _normalize(c) == "primarykey"), None)

                if entity_pk_col is None:
                    ri_issues.append({
                        "severity": "warning",
                        "rule": f"mapping_{side}_entity_primarykey_missing",
                        "file": mf,
                        "message": f"Resolved {side} entity type '{etype}' has no PrimaryKey column in CSV",
                        "details": {f"{side}Type": etype},
                    })
                    continue

                etype_sql = str(etype).replace("'", "''")

                broken_df = spark.sql(f"""
                    SELECT m.{pk_col}, m.{type_col}
                    FROM   {STAGING_DB}.{mtbl}  m
                    LEFT   JOIN {STAGING_DB}.{etbl} e
                           ON e.`{entity_pk_col}` = m.{pk_col}
                    WHERE  m.{type_col} = '{etype_sql}'
                      AND  e.`{entity_pk_col}` IS NULL
                      AND  m.{pk_col} IS NOT NULL
                """)
                broken_count = broken_df.count()
                if broken_count > 0:
                    samples = [r[pk_col] for r in broken_df.limit(5).collect()]
                    ri_issues.append({
                        "severity": "error",
                        "rule": f"mapping_{side}_fk_broken",
                        "file": mf,
                        "message": f"{broken_count} broken {side} FK(s) for entity type '{etype}'",
                        "details": {
                            f"{side}Type": etype,
                            "brokenCount": broken_count,
                            "samples": samples,
                        },
                    })
                    print(f"  ❌ {broken_count} broken {side} FK(s) for '{etype}' in {Path(mf).name}")
                else:
                    print(f"  ✓ {side} FK OK for '{etype}' in {Path(mf).name}")

    # 6) Build report in same format as Python mode
    severity_counts = {"error": 0, "warning": 0, "info": 0}
    for issue in ri_issues:
        sev = issue.get("severity", "warning").lower()
        severity_counts[sev] = severity_counts.get(sev, 0) + 1

    report = {
        "generatedAtUtc": datetime.now(timezone.utc).isoformat(),
        "riMode": "sql",
        "summary": severity_counts,
        "issues": ri_issues,
    }

    # 7) Save report
    out_path = Path(OUTPUT_JSON)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(_json.dumps(report, indent=2), encoding="utf-8")
    print(f"\nReport saved to: {OUTPUT_JSON}")

else:
    raise ValueError(f"Unknown RI_MODE: '{RI_MODE}'. Use 'python' or 'sql'.")

print(f"\nReport saved to: {OUTPUT_JSON}")
print_report_summary(report)

## 4) Inspect Top Issues

In [ ]:
issues = report.get("issues", [])
print(f"Total issues: {len(issues)}")
for i, issue in enumerate(issues[:25], start=1):
    row = issue.get("row")
    row_part = f":{row}" if row is not None else ""
    print(f"{i:02d}. [{issue.get('severity', 'warning').upper()}] {issue.get('rule')} - {issue.get('file')}{row_part}")
    print(f"    {issue.get('message')}")

In [ ]:
import json as _json

# Where is the full report stored?
report_path = Path(OUTPUT_JSON)
if report_path.exists():
    size_kb = report_path.stat().st_size / 1024
    print(f"Report file:  {report_path}")
    print(f"Size:         {size_kb:.1f} KB")
    print(f"Download via: Fabric Lakehouse Files -> isa95_quality_output/quality-check-report.json")
else:
    print(f"Report not yet written to disk: {OUTPUT_JSON}")

In [ ]:
import csv
import re
from collections import defaultdict
from pathlib import Path

# Matches timestamp/shard suffixes in generated file names.
FILENAME_TIMESTAMP_SUFFIX_RE = re.compile(
    r"^(?P<base>.+?)(?:[_\-. ]+)(?:"
    r"\d{8}T\d{6}Z?"
    r"|\d{8}[_\-. ]\d{6}Z?"
    r"|\d{14}"
    r"|\d{8}"
    r"|\d{4}-\d{2}-\d{2}(?:[_\-. ]\d{2}(?::?\d{2}){1,2})?"
    r")(?:[_\-. ]+\d{1,4})?$",
    re.IGNORECASE,
)


def strip_filename_timestamp_suffix(stem: str) -> str:
    current = stem.strip()
    for _ in range(3):
        m = FILENAME_TIMESTAMP_SUFFIX_RE.match(current)
        if not m:
            break
        nxt = m.group("base").strip(" _-.")
        if not nxt or nxt == current:
            break
        current = nxt
    return current


def normalize_entity_label(stem: str) -> str:
    base = strip_filename_timestamp_suffix(stem)
    return base.replace("_", " ").strip()


def print_markdown_table(headers, rows):
    print("| " + " | ".join(headers) + " |")
    print("| " + " | ".join(["---"] * len(headers)) + " |")
    for row in rows:
        print("| " + " | ".join(row) + " |")


def write_csv_table(output_path: Path, headers, rows):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(headers)
        writer.writerows(rows)


print("Entity and Mapping Row Counts (copy-friendly tables)")

entity_file_rows = {}
entity_summary_rows = defaultdict(int)
mapping_file_rows = {}
mapping_summary_rows = defaultdict(int)
mapping_summary_files = defaultdict(list)

for csv_file in Path(DATA_DIR).rglob("*.csv"):
    try:
        with open(csv_file, "r", encoding="utf-8-sig", newline="") as f:
            row_count = max(sum(1 for _ in csv.reader(f)) - 1, 0)

        stem = csv_file.stem
        lower_stem = stem.lower()

        # Mapping files expected pattern: <source>_to_<target>_mapping[_timestamp][_shard]
        if "_to_" in lower_stem and "mapping" in lower_stem:
            mapping_file_rows[csv_file.name] = row_count

            no_suffix = strip_filename_timestamp_suffix(stem)
            no_suffix_lower = no_suffix.lower()
            if no_suffix_lower.endswith("_mapping"):
                no_suffix = no_suffix[: -len("_mapping")]

            parts = no_suffix.split("_to_", 1)
            if len(parts) == 2:
                src = normalize_entity_label(parts[0])
                tgt = normalize_entity_label(parts[1])
                mapping_key = f"{src} -> {tgt}"
            else:
                mapping_key = normalize_entity_label(no_suffix)

            mapping_summary_rows[mapping_key] += row_count
            mapping_summary_files[mapping_key].append(csv_file.name)
        else:
            entity_file_rows[csv_file.name] = row_count
            entity_name = normalize_entity_label(stem)
            entity_summary_rows[entity_name] += row_count

    except Exception as exc:
        print(f"Warning: failed to read {csv_file.name}: {exc}")


# 1) Detailed entity files
print("\nDetailed entity file counts")
entity_detail_rows = [
    [name, f"{count:,}"]
    for name, count in sorted(entity_file_rows.items())
]
entity_detail_rows.append(["TOTAL", f"{sum(entity_file_rows.values()):,}"])
print_markdown_table(["Entity File", "Rows"], entity_detail_rows)

# 2) Consolidated entity summary (requested)
print("\nConsolidated entity counts")
entity_summary_table_rows = [
    [name, f"{count:,}"]
    for name, count in sorted(entity_summary_rows.items())
]
entity_summary_table_rows.append(["TOTAL", f"{sum(entity_summary_rows.values()):,}"])
print_markdown_table(["Entity", "Rows"], entity_summary_table_rows)

# 3) Detailed mapping files
print("\nDetailed mapping file counts")
mapping_detail_rows = [
    [name, f"{count:,}"]
    for name, count in sorted(mapping_file_rows.items())
]
mapping_detail_rows.append(["TOTAL", f"{sum(mapping_file_rows.values()):,}"])
print_markdown_table(["Mapping File", "Rows"], mapping_detail_rows)

# 4) Consolidated mapping summary
print("\nConsolidated mapping counts")
mapping_summary_table_rows = [
    [key, f"{count:,}", str(len(mapping_summary_files[key]))]
    for key, count in sorted(mapping_summary_rows.items())
]
mapping_summary_table_rows.append(["TOTAL", f"{sum(mapping_summary_rows.values()):,}", "-"])
print_markdown_table(["Mapping (Source -> Target)", "Rows", "Files"], mapping_summary_table_rows)

# 5) Persist all count tables as CSV for download/use outside notebook
counts_output_dir = Path(OUTPUT_JSON).parent / "counts"

entity_detail_csv_rows = [[r[0], r[1].replace(",", "")] for r in entity_detail_rows]
entity_summary_csv_rows = [[r[0], r[1].replace(",", "")] for r in entity_summary_table_rows]
mapping_detail_csv_rows = [[r[0], r[1].replace(",", "")] for r in mapping_detail_rows]
mapping_summary_csv_rows = [[r[0], r[1].replace(",", ""), r[2]] for r in mapping_summary_table_rows]

entity_detail_csv = counts_output_dir / "entity_file_counts.csv"
entity_summary_csv = counts_output_dir / "entity_consolidated_counts.csv"
mapping_detail_csv = counts_output_dir / "mapping_file_counts.csv"
mapping_summary_csv = counts_output_dir / "mapping_consolidated_counts.csv"

write_csv_table(entity_detail_csv, ["Entity File", "Rows"], entity_detail_csv_rows)
write_csv_table(entity_summary_csv, ["Entity", "Rows"], entity_summary_csv_rows)
write_csv_table(mapping_detail_csv, ["Mapping File", "Rows"], mapping_detail_csv_rows)
write_csv_table(mapping_summary_csv, ["Mapping (Source -> Target)", "Rows", "Files"], mapping_summary_csv_rows)

print("\nCSV outputs written:")
print(f"- {entity_detail_csv}")
print(f"- {entity_summary_csv}")
print(f"- {mapping_detail_csv}")
print(f"- {mapping_summary_csv}")

In [ ]:
# ── Full Report Explorer ───────────────────────────────────────────────────
# Adjust filters to drill into specific issues.

FILTER_SEVERITY = None          # None = all | "error" | "warning" | "info"
FILTER_RULE     = None          # None = all | e.g. "mapping_source_fk_broken"
FILTER_FILE     = None          # None = all | partial filename match, e.g. "Equipment"
MAX_DISPLAY     = 200           # max issues to display (set to None for all)
GROUP_BY_RULE   = True          # show grouped counts first

issues_all = report.get("issues", [])

# Apply filters
filtered = issues_all
if FILTER_SEVERITY:
    filtered = [i for i in filtered if i.get("severity", "").lower() == FILTER_SEVERITY.lower()]
if FILTER_RULE:
    filtered = [i for i in filtered if i.get("rule", "").lower() == FILTER_RULE.lower()]
if FILTER_FILE:
    filtered = [i for i in filtered if FILTER_FILE.lower() in (i.get("file") or "").lower()]

print(f"Total issues in report : {len(issues_all)}")
print(f"Matching current filter: {len(filtered)}")

if GROUP_BY_RULE:
    from collections import Counter
    counts = Counter(
        f"[{i.get('severity','?').upper()}] {i.get('rule','?')}"
        for i in filtered
    )
    print("\n── Issues by Rule ────────────────────────────────────────────────")
    for rule_key, count in sorted(counts.items(), key=lambda x: -x[1]):
        print(f"  {count:6d}  {rule_key}")

display_items = filtered[:MAX_DISPLAY] if MAX_DISPLAY else filtered
print(f"\n── Detail ({len(display_items)} shown) ──────────────────────────────────────────")
for i, issue in enumerate(display_items, start=1):
    row = issue.get("row")
    row_part = f":{row}" if row is not None else ""
    file_short = Path(issue.get("file", "")).name or issue.get("file", "")
    print(f"\n{i:04d}. [{issue.get('severity','?').upper()}] {issue.get('rule')}")
    print(f"      File   : {file_short}{row_part}")
    print(f"      Message: {issue.get('message')}")
    details = issue.get("details")
    if details:
        print(f"      Details: {_json.dumps(details, ensure_ascii=False)}")

## 5) Subsequent Fast Runs
After cache is built, rerun without DTDL refresh.

In [ ]:
fast_report = run_validation_for_notebook(
    data_dirs=DATA_DIRS,
    metadata_cache=METADATA_CACHE,
    refresh_metadata=False,
    output_json=OUTPUT_JSON,
    include_master_process=False,
    max_rows_per_file=MAX_ROWS_PER_FILE,
)

print_report_summary(fast_report)